### Another Experiment

In [26]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, TargetEncoder, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, auc, roc_curve, roc_auc_score

In [2]:
train = pd.read_csv('../Data/train.csv')
test = pd.read_csv('../Data/test.csv')

### Data Exploration

In [3]:
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


#### Missing Values?

In [4]:
train.isna().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64

In [5]:
test.isna().sum()

id                        0
Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
dtype: int64

#### Data info

In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

#### Dropping unnecessary columns

In [7]:
train.drop(['id'], axis = 1, inplace = True)

#### Checking imbalance in dataset

In [8]:
train['PitNextLap'].value_counts()

# The data is imbalanced but not extremely:
# PitNextLap
# 0.0    351759
# 1.0     87381
# 80/20 split imbalance

PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64

### Feature Engineering

In [9]:
train.columns

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')

In [10]:
#A tyre stint in motorsport is the continuous, uninterrupted period a car spends on track between pit stops, ranging from when it leaves the pits (or starts the race) until it returns

train['TyreLife_per_Stint'] = train.apply(
    lambda x: x['TyreLife'] / x['Stint'] if x['Stint'] != 0 else x['TyreLife'], axis = 1
)
test['TyreLife_per_Stint'] = test.apply(
    lambda x: x['TyreLife'] / x['Stint'] if x['Stint'] != 0 else x['TyreLife'], axis = 1
)


#Cumulative degradation is the progressive, combined loss of performance, functionality, or quality in a system caused by the accumulation of multiple stressors, damage mechanisms, or aging over time

train['Tyre_Wear'] = train['TyreLife'] * train['Cumulative_Degradation']

test['Tyre_Wear'] = test['TyreLife'] * test['Cumulative_Degradation']

#In Formula 1, the pit window is the optimal range of laps for a driver to pit, balancing tire degradation, fuel load, and race position to minimize time lost 

train['Early_PitWindow'] = (train['RaceProgress'] < 0.3).astype(int)
train['Mid_PitWindow'] = ((train['RaceProgress'] >= 0.3 ) & (train['RaceProgress'] < 0.6)).astype(int)
train['Late_PitWindow'] = (train['RaceProgress'] > 0.6).astype(int)

test['Early_PitWindow'] = (test['RaceProgress'] < 0.3).astype(int)
test['Mid_PitWindow'] = ((test['RaceProgress'] >= 0.3 ) & (train['RaceProgress'] < 0.6)).astype(int)
test['Late_PitWindow'] = (test['RaceProgress'] > 0.6).astype(int)

#Likely a last Stint(use)

train['IsLastStint'] = (train['Stint'] > 3).astype(int)

test['IsLastStint'] = (test['Stint'] > 3).astype(int)

train = train.sort_values(['Driver', 'Race', 'Year', 'LapNumber'])
train['RollingMean_Laptime'] = (
    train.groupby(['Driver', 'Year', 'Race'])['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods = 1).mean())
)
train['LapTime_Trend'] = train['LapTime (s)'] - train['RollingMean_Laptime']

test = test.sort_values(['Driver', 'Race', 'Year', 'LapNumber'])
test['RollingMean_Laptime'] = (
    test.groupby(['Driver', 'Year', 'Race'])['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods = 1).mean())
)
test['LapTime_Trend'] = test['LapTime (s)'] - test['RollingMean_Laptime']


### Splitting Categorical and Numerical columns

In [11]:
num_cols = train.select_dtypes(include = 'number').columns.drop(['PitNextLap']).tolist()
cat_cols = train.select_dtypes(include = 'object').columns.tolist()

In [12]:
num_cols

['Year',
 'PitStop',
 'LapNumber',
 'Stint',
 'TyreLife',
 'Position',
 'LapTime (s)',
 'LapTime_Delta',
 'Cumulative_Degradation',
 'RaceProgress',
 'Position_Change',
 'TyreLife_per_Stint',
 'Tyre_Wear',
 'Early_PitWindow',
 'Mid_PitWindow',
 'Late_PitWindow',
 'IsLastStint',
 'RollingMean_Laptime',
 'LapTime_Trend']

In [13]:
cat_cols

['Driver', 'Compound', 'Race']

#### Encoding Columns

In [14]:
X = train.drop(['PitNextLap'], axis = 1)
y = train['PitNextLap']

In [15]:
# Target Encoder
cat_pipe1 = Pipeline(steps = [
    ('TargetEncoder', TargetEncoder())
])

# One Hot Encoder
cat_pipe2 = Pipeline(steps = [
    ('OneHotEncoder', OneHotEncoder())
])

# Numerical Pipeline

num_pipe = Pipeline(steps = [
    ('Scaler', StandardScaler())
])

#### Transformer

In [16]:
preprocessor = ColumnTransformer(transformers = [
    ('Numerical', num_pipe, num_cols),
    ('Target', cat_pipe1, ['Driver', 'Race']),
    ('OHE', cat_pipe2, ['Compound'])
])

### Model Training

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [18]:
X_train_transformed = preprocessor.fit_transform(X_train, y_train)
X_test_transformed = preprocessor.transform(X_test)

In [19]:
models = {
    'LogisticRegression': LogisticRegression(class_weight = 'balanced'),
    'RandomForestClassifier': RandomForestClassifier(class_weight = 'balanced'),
    'DecisonTreeClassifier': DecisionTreeClassifier(class_weight = 'balanced'),
    'Catboost': CatBoostClassifier(verbose = False),
    'XGboost': XGBClassifier(scale_pos_weight = 4.02)
}

In [20]:
best_model = None
best_score = 0
best_name = ''

for name, model in models.items():
    model.fit(X_train_transformed, y_train)
    predict = model.predict(X_test_transformed)

    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test_transformed)[:, 1]
    elif hasattr(model, 'decision_function'):
        proba = model.decision_function(X_test_transformed)
    else:
        continue

    print(f'Model: {name}')
    print('*' * 50)
    print(f'Classification Report:\n{classification_report(predict, y_test)}')

    score = roc_auc_score(y_test, proba)
    if score > best_score:
        best_score = score
        best_model = model
        best_name = name


Model: LogisticRegression
**************************************************
Classification Report:
              precision    recall  f1-score   support

         0.0       0.80      0.94      0.86     59845
         1.0       0.80      0.50      0.61     27983

    accuracy                           0.80     87828
   macro avg       0.80      0.72      0.74     87828
weighted avg       0.80      0.80      0.78     87828

Model: RandomForestClassifier
**************************************************
Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      0.92      0.93     72358
         1.0       0.67      0.76      0.71     15470

    accuracy                           0.89     87828
   macro avg       0.81      0.84      0.82     87828
weighted avg       0.90      0.89      0.89     87828

Model: DecisonTreeClassifier
**************************************************
Classification Report:
              precision    recall  f1-s

### Hyper-Parameter tuning

In [25]:
print('Best Model: ', best_name)

Best Model:  Catboost


In [27]:
cat = CatBoostClassifier(
    auto_class_weights = 'Balanced',
    eval_metric = 'AUC',
    random_seed = 7,
    verbose = 0
)

In [ ]:
gfk = GroupKFold(n_splits = 5)
groups = X_train['Race']

params = {
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [500, 1000, 1500],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [1, 3, 5, 7],
    'bagging_temperature': [0, 0.5, 1],
    'border_count': [32, 64, 128]
}

search = RandomizedSearchCV(
    estimator = cat,
    param_distributions = params,
    n_iter = 30,
    cv = gfk.split(X_train_transformed, y_train, groups = groups),
    n_jobs = -1,
    verbose = 1,
    random_state = 7
)

search.fit(X_train_transformed, y_train)

print('Best_param: ', search.best_params_)
print('Best AUC: ', search.best_score_)

In [ ]:
test_transformed = preprocessor.transform(test)
prediction = best_model.predict_proba(test_transformed)[:, 1]

### Submission

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'PitNextLap': prediction 
})

submission.to_csv('../Data/submissions/submission.csv', index = False)